# STEP 1: Input Sequences and clean

In [ ]:
dna_fs = input("Enter the fusion sequence: ")
gene_a_cds = input("Enter Gene A CDS sequence: ")
gene_b_cds = input("Enter Gene B CDS sequence: ")

gene_a = gene_a_cds.replace(" ", "").upper()
gene_b = gene_b_cds.replace(" ", "").upper()
dna_fs = dna_fs.replace(" ", "").upper()

# Step 2: Split the fusion sequence and search for each fragment¶

In [ ]:
left_fragment, right_fragment = dna_fs.split("*")

def search(fragment, gene):

    position = gene.find(fragment)

    if position == -1:
        return "Cant find"

    return position


print("\nLeft Fragment")
print("Gene A:", search(left_fragment, gene_a))
print("Gene B:", search(left_fragment, gene_b))

print("\nRight Fragment")
print("Gene A:", search(right_fragment, gene_a))
print("Gene B:", search(right_fragment, gene_b))


left_in_a = gene_a.find(left_fragment)
left_in_b = gene_b.find(left_fragment)

right_in_a = gene_a.find(right_fragment)
right_in_b = gene_b.find(right_fragment)


In [ ]:
# Step 4: Reconstruct Fusion CDS, save Junction

if left_in_a != -1 and right_in_b != -1:

    left_sequence = gene_a[:left_in_a + len(left_fragment)]
    right_sequence = gene_b[right_in_b:]

elif left_in_b != -1 and right_in_a != -1:

    left_sequence = gene_b[:left_in_b + len(left_fragment)]
    right_sequence = gene_a[right_in_a:]

else:

    print("\nUnable to reconstruct the fusion CDS.")
    exit()


fusion_cds = left_sequence + right_sequence

print("\nFusion CDS:\n")
print(fusion_cds)

junction = len(left_sequence)

print("\nDNA junction position:", junction)
print("Reading frame:", junction % 3)


# STEP 3 - Translate fusion CDS

In [ ]:
genetic_code = {
    "ATA":"I","ATC":"I","ATT":"I","ATG":"M",
    "ACA":"T","ACC":"T","ACG":"T","ACT":"T",
    "AAC":"N","AAT":"N","AAA":"K","AAG":"K",
    "AGC":"S","AGT":"S","AGA":"R","AGG":"R",
    "CTA":"L","CTC":"L","CTG":"L","CTT":"L",
    "CCA":"P","CCC":"P","CCG":"P","CCT":"P",
    "CAC":"H","CAT":"H","CAA":"Q","CAG":"Q",
    "CGA":"R","CGC":"R","CGG":"R","CGT":"R",
    "GTA":"V","GTC":"V","GTG":"V","GTT":"V",
    "GCA":"A","GCC":"A","GCG":"A","GCT":"A",
    "GAC":"D","GAT":"D","GAA":"E","GAG":"E",
    "GGA":"G","GGC":"G","GGG":"G","GGT":"G",
    "TCA":"S","TCC":"S","TCG":"S","TCT":"S",
    "TTC":"F","TTT":"F","TTA":"L","TTG":"L",
    "TAC":"Y","TAT":"Y","TAA":"*","TAG":"*","TGA":"*"
}


def translate(dna):

    protein = ""

    for i in range(0, len(dna)-2, 3):

        codon = dna[i:i+3]

        protein += genetic_code.get(codon, "X")

    return protein


protein = translate(fusion_cds)

gene_a_protein = translate(gene_a_cds)

gene_b_protein = translate(gene_b_cds)

junction_aa = junction // 3


print("\nFusion protein\n")

print(protein)

print("\nDNA junction:", junction)

print("AA junction:", junction_aa)

print("AA at junction:", protein[junction_aa])

# Step 4- Decide whether the junction is in-frame

In [ ]:
print("\nFrame remainder:", junction % 3)

if junction % 3 == 0:

    calculated_frame = "in-frame"

else:

    calculated_frame = "out-of-frame"

print("Calculated frame:", calculated_frame)



# Step 5- Generate all peptides

In [ ]:
window_size = int(input("Peptide length: "))

all_peptides = []

for start in range(len(protein)-window_size+1):

    peptide = protein[start:start+window_size]

    end = start + window_size - 1

    all_peptides.append((start,end,peptide))


# Step 6- Junction peptides

In [ ]:
junction_peptides = []

for start,end,peptide in all_peptides:

    if start <= junction_aa <= end:

        junction_peptides.append((start,peptide))

##Downstream peptides (ONLY for out-of-frame)
downstream_peptides = []

if calculated_frame == "out-of-frame":

    for start,end,peptide in all_peptides:

        if start >= junction_aa:

            downstream_peptides.append((start,peptide))



In [ ]:
# Step 7- Novelty check

In [ ]:
novel_peptides = []

for start,peptide in junction_peptides:

    in_gene_a = peptide in gene_a_protein

    in_gene_b = peptide in gene_b_protein

    if not in_gene_a and not in_gene_b:

        novel_peptides.append((start,peptide,"junction"))


for start,peptide in downstream_peptides:

    in_gene_a = peptide in gene_a_protein

    in_gene_b = peptide in gene_b_protein

    if not in_gene_a and not in_gene_b:

        novel_peptides.append((start,peptide,"downstream"))



In [ ]:
#Step 9- Dispay resuts

In [ ]:
print("\nNovel peptides\n")

for start,peptide,kind in novel_peptides:

    print(
        "AA",start,
        "|",
        kind,
        "|",
        peptide
    )

# Step 10 - Check whether Gene B is entered in the same frame

In [ ]:
# Amino acids contributed by Gene B in the fusion
fusion_gene_b = protein[junction_aa:]

# Junction position in Gene B CDS
gene_b_junction = len(gene_b_cds) - len(right_sequence)

gene_b_junction_aa = gene_b_junction // 3

# Native Gene B amino acids from the breakpoint onward
native_gene_b = gene_b_protein[gene_b_junction_aa:]

print("\nChecking reading frame...\n")

# Compare the first 30 amino acids
comparison_length = min(30, len(fusion_gene_b), len(native_gene_b))

matches = 0

for i in range(comparison_length):

    if fusion_gene_b[i] == native_gene_b[i]:

        matches += 1

identity = matches / comparison_length

print("Identity:", round(identity * 100, 2), "%")

if identity >= 0.90:

    calculated_frame = "in-frame"

else:

    calculated_frame = "out-of-frame"

print("Calculated frame:", calculated_frame)


# Step 11- Compare with Arriba/FusionCatcher


In [ ]:
reported_frame = row["Predicted_effect"]

print("Reported:", reported_frame)

print("Calculated:", calculated_frame)

if reported_frame.lower() == calculated_frame.lower():

    print("✓ Frame agrees with fusion caller.")

else:

    print("Frame does NOT agree with fusion caller.")

In [ ]:
# STEP 12- Annotate every peptide

In [ ]:
annotated_peptides = []

for start, peptide, kind in novel_peptides:

    annotated_peptides.append({

        "GeneA": gene_a,

        "GeneB": gene_b,

        "Reported_frame": reported_frame,

        "Calculated_frame": calculated_frame,

        "Junction_type": kind,

        "Peptide": peptide,

        "Length": len(peptide),

        "AA_position": start

    })

# Step 13 - Print

In [ ]:
print("\nCandidate neoantigen peptides\n")

for peptide in annotated_peptides:

    print(peptide)

# Step 14- Save every peptide

In [ ]:
results = []

In [ ]:
annotated_peptides.append({

    "GeneA": gene_a,

    "GeneB": gene_b,

    "Reported_frame": reported_frame,

    "Calculated_frame": calculated_frame,

    "Junction_type": kind,

    "Peptide": peptide,

    "Length": len(peptide),

    "AA_position": start

})




In [ ]:
results.append({

    "GeneA": gene_a,

    "GeneB": gene_b,

    "Reported_frame": reported_frame,

    "Calculated_frame": calculated_frame,

    "Junction_type": kind,

    "Peptide": peptide,

    "Length": len(peptide),

    "AA_position": start

})

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)

results_df.to_csv(
    "Fusion_candidate_peptides.csv",
    index=False
)

print("\nResults saved to Fusion_candidate_peptides.csv")

# Step 15- Save every resut 

In [ ]:
for start, peptide, kind in novel_peptides:

    results.append({

        "GeneA": gene_a,

        "GeneB": gene_b,

        "Reported_frame": reported_frame,

        "Calculated_frame": calculated_frame,

        "Peptide_type": kind,

        "Peptide": peptide,

        "Peptide_length": len(peptide),

        "AA_position": start,

        "Fusion_sequence": fusion

    })

# Step 16 - Export after All fusions are processed

In [ ]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    "Fusion_candidate_peptides.csv",
    index=False
)

print("\nFinished.")

print("Number of candidate peptides:", len(results_df))

print("Results saved as Fusion_candidate_peptides.csv")